In [0]:
#enforcing Schema
from pyspark.sql.types import *

silver_schema = StructType([
    StructField("transID",          StringType(),  nullable=False), # every trip must have an ID
    StructField("payCardID",        LongType(),    nullable=False), # every trip must have a card
    StructField("payCardBank",      StringType(),  nullable=True),
    StructField("payCardName",      StringType(),  nullable=True),
    StructField("payCardSex",       StringType(),  nullable=True),
    StructField("payCardBirthDate", IntegerType(), nullable=True),
    StructField("corridorID",       StringType(),  nullable=True),
    StructField("corridorName",     StringType(),  nullable=True),
    StructField("direction",        IntegerType(),  nullable=True),
    StructField("tapInStops",       StringType(),  nullable=True),
    StructField("tapInStopsName",   StringType(),  nullable=True),
    StructField("tapInStopsLat",    DoubleType(),  nullable=True),
    StructField("tapInStopsLon",    DoubleType(),  nullable=True),
    StructField("stopStartSeq",     IntegerType(), nullable=True),
    StructField("tapInTime",        TimestampType(),nullable=False), # must have tap in time
    StructField("tapOutStops",      StringType(),  nullable=True),  # can be missing — flagged
    StructField("tapOutStopsName",  StringType(),  nullable=True),
    StructField("tapOutStopsLat",   DoubleType(),  nullable=True),
    StructField("tapOutStopsLon",   DoubleType(),  nullable=True),
    StructField("stopEndSeq",       DoubleType(),  nullable=True),
    StructField("tapOutTime",       TimestampType(),nullable=True), # can be missing — flagged
    StructField("payAmount",        DoubleType(),  nullable=True)
])

In [0]:
#reading the dataframe from the bronze schema
df = spark.read.format("delta") \
    .schema(silver_schema) \
    .table("transjakarta_dataset.bronze.transjakarta_raw")
display(df)

In [0]:
from pyspark.sql import functions as F

# Remove duplicates
df = df.dropDuplicates()


# Check null distributions
display(df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]))

TAP IN STOPS CLEANING

In [0]:
#This is for checking if the tapInStops is related to tapInStopsName which wil help on inputting tapInStops value
tapInStops = df.select("tapInStops", "tapInStopsName") \
                .dropna() \

# Check for tapInStops with more than one tapInStopsName
dup_check_tapInStops = tapInStops.groupBy("tapInStops", "tapInStopsName") \
    .count() \
    .filter(F.col("count") > 1)

display(dup_check_tapInStops)

#the result is resulting that every tapInStops id is related to tapInStopsName

In [0]:
from pyspark.sql.window import Window

#using window function to fill the missing tapInStops value based on tapInStopsName
window_spec = Window.partitionBy("tapInStopsName")
df_filled = df.withColumn(
    "tapInStops",
    F.coalesce(
        F.col("tapInStops"),
        F.max("tapInStops").over(window_spec)
    )
)

display(df_filled.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_filled.columns]))


CORRIDOR ID CLEANING

In [0]:
#This is for checking if the Corridor id is related to CorridorName which wil help on inputting tapInStops value
corridorCheck = df.select("corridorID", "corridorName") \
                .dropna() \

# Check for Corridor with more than one CorridorName
dup_check_corridor = corridorCheck.groupBy("corridorID", "corridorName") \
    .count() \
    .filter(F.col("count") > 1)

display(dup_check_corridor)

#the result is resulting that every Corridor id is related to CorridorName

In [0]:
#using window function to fill the missing corridorName value based on CorridorId
window_spec = Window.partitionBy("corridorID")
df_filled = df_filled.withColumn(
    "corridorName",
    F.coalesce(
        F.col("corridorName"),
        F.max("corridorName").over(window_spec)
    )
)

display(df_filled.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_filled.columns]))

In [0]:
#using window function to fill the missing CorridorId value based on CorridorName
window_spec = Window.partitionBy("corridorName")
df_filled = df_filled.withColumn(
    "corridorID",
    F.coalesce(
        F.col("corridorID"),
        F.max("corridorID").over(window_spec)
    )
)

display(df_filled.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_filled.columns]))

CORRIDOR CLEANING

In [0]:
#This is for checking if the tapOutStops is related to tapOutStopsName which wil help on inputting tapInStops value
tapOutStopsCheck = df.select("tapOutStops", "tapOutStopsName") \
                .dropna() \

# Check for tapOutStops with more than one tapOutStopsName
dup_check_tapOut = tapOutStopsCheck.groupBy("tapOutStops", "tapOutStopsName") \
    .count() \
    .filter(F.col("count") > 1)

display(dup_check_tapOut)

#the result is resulting that every tapOutStops id is related to tapOutStopsName

In [0]:
#using window function to fill the missing tapOutStops value based on tapOutStopsName
window_spec = Window.partitionBy("tapOutStopsName")
df_filled = df_filled.withColumn(
    "tapOutStops",
    F.coalesce(
        F.col("tapOutStops"),
        F.max("tapOutStops").over(window_spec)
    )
)

display(df_filled.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_filled.columns]))

In [0]:
#using window function to fill the missing tapOutStopsName value based on tapOutStops
window_spec = Window.partitionBy("tapOutStops")
df_filled = df_filled.withColumn(
    "tapOutStopsName",
    F.coalesce(
        F.col("tapOutStopsName"),
        F.max("tapOutStopsName").over(window_spec)
    )
)

display(df_filled.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_filled.columns]))

In [0]:
#using window function to fill the missing tapOutStops value based on tapOutStopsName
window_spec = Window.partitionBy("tapOutStopsName")
df_filled = df_filled.withColumn(
    "tapOutStops",
    F.coalesce(
        F.col("tapOutStops"),
        F.max("tapOutStops").over(window_spec)
    )
)

display(df_filled.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_filled.columns]))

In [0]:
#Flag for missing tapInStops and tapOutStops
df_filled = df_filled.withColumn(
    "Note",
    F.when(F.col("tapOutStops").isNull(), "MissingTapOut")
     .when(F.col("tapInStops").isNull(), "MissingTapIn")
     .otherwise("tripSuccessful")
)

In [0]:
#printing the schema 
df_filled.printSchema()
display(df_filled.describe())

In [0]:
#Trim all the string column
from pyspark.sql.types import StringType
string_cols = [f.name for f in df_filled.schema.fields if isinstance(f.dataType, StringType)]

for col_name in string_cols:
    df_filled = df_filled.withColumn(col_name, F.trim(F.col(col_name)))

In [0]:
#Checking if the transID column need more standardisation
display(df_filled.select("transID"))
#And it didnt need anymore standardisation

In [0]:
#Checking if the payCardID column need more standardisation
display(df_filled.select("payCardID"))
#And it didnt need anymore standardisation

In [0]:
#Checking if the payCardBank column need more standardisation
df_filled.select("payCardBank").distinct().show()
#And it didnt need anymore standardisation

In [0]:
#Checking if the payCardName column need more standardisation
display(df_filled.select("payCardName"))
#And it didnt need anymore standardisation

In [0]:
#Checking if the payCardSex column need more standardisation
df_filled.select("payCardSex").distinct().show()
#And it didnt need anymore standardisation

In [0]:
#Checking if the payCardBirthDate column need more standardisation
display(df_filled.select("payCardBirthDate").distinct)
#And it didnt need anymore standardisation

In [0]:
#Checking if the corridorName column need more standardisation
display(df_filled.select("corridorName"))
#And it didnt need anymore standardisation

In [0]:
#Checking if the direction column need more standardisation
from pyspark.sql.types import IntegerType
df_filled = df_filled.withColumn("direction", F.col("direction").cast(IntegerType()))
df_filled.select("direction").distinct().show()
#And it didnt need anymore standardisation


In [0]:
#Checking if if the tapInStops Column need more standardisation by checking if every value is already full capital
df_filled.filter(
    F.col("tapInStops") != F.upper(F.col("tapInStops"))
).select("tapInStops").show(20,truncate=False)
#And it didnt need anymore standardisation

In [0]:
#Checking if the direction tapInStopsName need more standardisation
display(df_filled.select("tapInStopsName"))
#And it didnt need anymore standardisation because i cant invents any name in silver, i just trim it and done.

In [0]:
#Checking if the direction tapInStopsLon,tapInStopsLat,and stopStartSeq need more standardisation
display(df_filled.select("tapInStopsLon","tapInStopsLat","stopStartSeq"))
# and it needs more standarndisation cause it could risk changing the crucial data

In [0]:
#Checking if the tapInTime column need more standardisation 
display(df_filled.select("tapInTime"))
#and it isnt needed anymore because the column is already timestamp datatype and using proper format

In [0]:
#Checking if the tapOutStops,tapOutStopsName,tapOutStopsLat, stopEndSeq, and tapOutStopsLon column need more standardisation 
display(df_filled.select("tapOutStops","tapOutStopsName","tapOutStopsLat","tapOutStopsLon","stopEndSeq"))
#and it isnt needed anymore 

In [0]:
#Checking if the tapOutTime and payAmount column need more standarndisation
display(df_filled.select("tapOutTime","payAmount"))
#and it didnt need more standardisation

In [0]:
df_filled = df_filled.withColumn("silver_processed_at", F.current_timestamp()) \
            .withColumn("silver_processed_by", F.lit("bronze_to_silver"))

In [0]:
#Writing the df as delta and save it as a table and put it into the silver schema
df_filled.write.format("delta").mode("overwrite").saveAsTable("transjakarta_dataset.silver.transjakarta_raw")